# Week 2 Day 2 — LangChain Agent

Rebuild Day 1’s agent with LangChain: tools, AgentExecutor, memory, structured output.
LLM backend: **Groq** via `langchain-groq` (same free-key setup as Day 1).

## Setup

```powershell
pip install -r requirements.txt
```

`GROQ_API_KEY` in `.env` (Day-2 or Day-1).

In [1]:
from langchain_agent import (
    TOOLS,
    MODEL,
    concept_map,
    build_lcel_chain,
    build_agent_executor,
    build_agent_with_memory,
    clear_session,
    recommend_structured,
    get_llm,
    flaky_divide,
)
import json

print("Model:", MODEL)
print("Tools:", [t.name for t in TOOLS])

Model: openai/gpt-oss-20b
Tools: ['calculator', 'get_weather', 'lookup_product', 'flaky_divide']


## Task 1 — Setup & core concepts

### Map to Day 1

In [2]:
for k, v in concept_map().items():
    print(f"{k}\n  → {v}\n")

LLM wrapper (ChatGroq)
  → Day 1: groq.Groq() + chat.completions.create

Tool (@tool)
  → Day 1: TOOLS JSON schemas + execute_tool()

AgentExecutor
  → Day 1: while-loop that appends tool results until final text

Memory (message history)
  → Day 1: messages list you managed by hand

agent_scratchpad
  → Day 1: intermediate tool_calls / observations in the loop



### LCEL: `prompt | llm | parser`

The pipe builds a **Runnable sequence**: each stage’s output feeds the next. LangChain implements a shared invoke/stream interface so you compose chains without nesting calls by hand.

In [3]:
chain = build_lcel_chain()
print(chain.invoke({"question": "In one short sentence, what makes software agentic?"}))

Software is agentic when it can perceive its environment, make decisions, and act autonomously to pursue goals.


## Task 2 — Tools

Three+ tools via `@tool`. **Docstrings are tool descriptions** — they are sent to the model as part of the tool schema, so clear docstrings improve tool choice (same role as Day 1 JSON `description` fields).

| Tool | Source |
|------|--------|
| `calculator` | Day 1 reuse |
| `get_weather` | Day 1 reuse |
| `lookup_product` | **New** — reads `data/products.csv` |
| `flaky_divide` | Error-handling demo |

In [4]:
for t in TOOLS:
    print("===", t.name, "===")
    print(t.description)
    print()

print(TOOLS[0].invoke({"expression": "10 + 5"}))
print(TOOLS[2].invoke({"query": "Wireless Mouse"})[:400])

=== calculator ===
Evaluate a basic arithmetic expression such as '79.5 - 18.99' or '(49.99+39.95)/2'.
Use this for price differences, totals, and other math. Do not invent numbers.

=== get_weather ===
Look up stub weather for one city. Known cities: Karachi, Lahore, Islamabad, London, Tokyo.
Returns temperature in Celsius and a short condition. Call once per city.

=== lookup_product ===
Search the local products.csv catalog by product name or product_id (e.g. 'Wireless Mouse' or 'P001').
Returns matching rows with price_usd and stock. Use this before comparing or recommending products.

=== flaky_divide ===
Divide a by b. This tool is intentionally unreliable for error-handling demos:
about half the time it raises a ZeroDivisionError or RuntimeError even when b != 0.
Prefer calculator for normal math; use this only when asked to test flaky_divide.

{"expression": "10 + 5", "result": 15.0}
{
  "query": "Wireless Mouse",
  "matches": [
    {
      "product_id": "P001",
      "name": "

## Task 3 — AgentExecutor + verbose trace

Multi-step run with `verbose=True`. In the log, mark:
- **Reason** — model plans / decides on a tool
- **Act** — tool call
- **Observe** — tool result

Compared to Day 1: same loop, but message formatting and scratchpad updates are inside LangChain.

In [5]:
agent = build_agent_executor(verbose=True)
multi = agent.invoke(
    {
        "input": (
            "Look up the price of Wireless Mouse and Mechanical Keyboard in the catalog. "
            "Use the calculator to get the price difference. Which is cheaper?"
        ),
        "chat_history": [],
    }
)
print("\n=== FINAL OUTPUT ===")
print(multi["output"])
print("\n=== INTERMEDIATE STEPS (Act / Observe pairs) ===")
for i, (action, observation) in enumerate(multi.get("intermediate_steps", []), 1):
    print(f"\nStep {i} ACT: {action.tool} {action.tool_input}")
    print(f"Step {i} OBSERVE: {str(observation)[:300]}")



> Entering new AgentExecutor chain...



Invoking: `lookup_product` with `{'query': 'Wireless Mouse'}`


{
  "query": "Wireless Mouse",
  "matches": [
    {
      "product_id": "P001",
      "name": "Wireless Mouse",
      "category": "accessories",
      "price_usd": "18.99",
      "in_stock": "true"
    }
  ]
}


Invoking: `lookup_product` with `{'query': 'Mechanical Keyboard'}`


{
  "query": "Mechanical Keyboard",
  "matches": [
    {
      "product_id": "P002",
      "name": "Mechanical Keyboard",
      "category": "accessories",
      "price_usd": "79.50",
      "in_stock": "true"
    }
  ]
}


Invoking: `calculator` with `{'expression': '79.50 - 18.99'}`


{"expression": "79.50 - 18.99", "result": 60.510000000000005}

**Prices**

| Product | Price (USD) |
|---------|-------------|
| Wireless Mouse | **$18.99** |
| Mechanical Keyboard | **$79.50** |

**Price difference**

$79.50 – $18.99 = $60.51

**Cheaper item**

The **Wireless Mouse** is cheaper than the Mechanical Keyboard.

> Finished chain.

=== FINAL OUTPUT ===
**Prices**

| Product | Price (USD) |
|---------|-------------|
| Wireless Mouse | **$18.99** |
| Mechanical Keyboard | **$79.50** |

**Price difference**

$79.50 – $18.99 = $60.51

**Cheaper item**

The **Wireless Mouse** is cheaper than the Mechanical Keyboard.

=== INTERMEDIATE STEPS (Act / Observe pairs) ===

Step 1 ACT: lookup_product {'query': 'Wireless Mouse'}
Step 1 OBSERVE: {
  "query": "Wireless Mouse",
  "matches": [
    {
      "product_id": "P001",
      "name": "Wireless Mouse",
      "category": "accessories",
      "price_usd": "18.99",
      "in_stock": "true"
    }
  ]
}

Step 2 ACT: lookup_product {'query': 'Mechanical Keyboard'}
Step 2 OBSERVE: {
  "query": "Mechanic

### Trace notes (fill from the verbose log above)

- **Reason:** model chooses `lookup_product` / `calculator` instead of guessing prices.
- **Act:** tool names + arguments in the log / `intermediate_steps`.
- **Observe:** CSV rows and calculator JSON come back as observations.
- **vs Day 1:** same ReAct shape; LangChain hides raw `tool_calls` message wiring unless you dig into verbose / intermediate steps.

## Task 4 — Memory (3-turn follow-ups)

`RunnableWithMessageHistory` keeps prior turns so later questions can say “compare to Y” without repeating X.

In [6]:
session = "budget-demo"
clear_session(session)
remember = build_agent_with_memory(verbose=False)
cfg = {"configurable": {"session_id": session}}

turns = [
    "Find the price of the Wireless Mouse in the catalog.",
    "Now compare it to the Mechanical Keyboard.",
    "Which one should I recommend to a budget-conscious client?",
]

for i, text in enumerate(turns, 1):
    out = remember.invoke({"input": text}, config=cfg)
    print(f"\n--- Turn {i}: {text}")
    print(out["output"])

C:\Users\DELL\AppData\Local\Temp\ipykernel_19552\1782187760.py:3: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  remember = build_agent_with_memory(verbose=False)



--- Turn 1: Find the price of the Wireless Mouse in the catalog.
The **Wireless Mouse** (product ID P001) is listed at **$18.99** and is currently in stock.



--- Turn 2: Now compare it to the Mechanical Keyboard.
**Price comparison**

| Product | Price (USD) |
|---------|-------------|
| Wireless Mouse | $18.99 |
| Mechanical Keyboard | $79.50 |

**Difference**  
The Mechanical Keyboard is **$60.51 more expensive** than the Wireless Mouse.



--- Turn 3: Which one should I recommend to a budget-conscious client?
For a budget‑conscious client, the **Wireless Mouse (P001)** is the better recommendation.  
- **Price**: $18.99 vs. $79.50 for the Mechanical Keyboard.  
- **Cost‑effectiveness**: It’s roughly 76 % cheaper, freeing up budget for other items or upgrades.  
- **Sufficient functionality**: A wireless mouse covers the essential needs of most users without the higher cost of a full mechanical keyboard.  

If the client later needs a keyboard, you can suggest a more affordable membrane or chiclet keyboard instead of the mechanical model.


## Task 5 — Structured output & error handling

### Structured recommendation (Pydantic)

In [7]:
rec = recommend_structured(
    "Compare Noise-Cancel Headset and Studio Headset. "
    "Recommend one for a budget-conscious client who still wants decent audio."
)
print(rec.model_dump_json(indent=2))

{
  "recommended_product": "Noise-Cancel Headset (P006)",
  "price_usd": 49.99,
  "compared_to": "Studio Headset (P007)",
  "reason": "decent audio at a low price",
  "in_stock": true
}


### Tool failures

`AgentExecutor(..., handle_tool_error=True)` turns exceptions into observations. `flaky_divide` raises often on purpose.

In [8]:
# Direct tool: show that it can throw
failures = 0
successes = 0
for _ in range(6):
    try:
        print(flaky_divide.invoke({"a": 10, "b": 2}))
        successes += 1
    except Exception as e:
        failures += 1
        print("raised:", type(e).__name__, e)
print(f"direct calls → success={successes}, fail={failures}")

# Through the agent: should recover gracefully (error text in the loop)
agent_err = build_agent_executor(verbose=True)
err_out = agent_err.invoke(
    {
        "input": "Call flaky_divide with a=10 and b=2. If it fails, say it failed and compute 10/2 with calculator instead.",
        "chat_history": [],
    }
)
print("\nAGENT RECOVERY OUTPUT:")
print(err_out["output"])

{"a": 10.0, "b": 2.0, "result": 5.0}
raised: RuntimeError Simulated tool failure inside flaky_divide
raised: RuntimeError Simulated tool failure inside flaky_divide
raised: RuntimeError Simulated tool failure inside flaky_divide
{"a": 10.0, "b": 2.0, "result": 5.0}
raised: RuntimeError Simulated tool failure inside flaky_divide
direct calls → success=2, fail=4




> Entering new AgentExecutor chain...



Invoking: `flaky_divide` with `{'a': 10, 'b': 2}`


{"a": 10.0, "b": 2.0, "result": 5.0}

The division succeeded: 10 ÷ 2 = **5.0**.

> Finished chain.

AGENT RECOVERY OUTPUT:
The division succeeded: 10 ÷ 2 = **5.0**.


### LangChain vs Day 1 (short)

LangChain made tools, the agent loop, memory, and structured answers much less boilerplate than the raw Groq loop. The tradeoff is abstraction leakiness: API names shift across versions, verbose logs become the main debugger, and the exact messages on the wire are harder to see. Day 1 was longer to write but fully transparent; Day 2 is faster to assemble once you trust the wrappers.

## Summary

Notebook covers Tasks 1–5. Companion files: `langchain_agent.py`, `data/products.csv`, `writeup.md`.